In [22]:
import pandas as pd

In [43]:
df_results = pd.read_csv("services/prompteng/output/postprocessed_output_qwen2.5_7b_autosave.csv")

/var/folders/4g/8j_z4q0n2zlftk4s8jmrj1sw0000gn/T/ipykernel_69219/3199770479.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df_results = pd.read_csv("services/prompteng/output/postprocessed_output_qwen2.5_7b_autosave.csv")


In [ ]:
import json

from prompteng.postprocessing import postprocess_json_output


df_results["qwen2.5:7b"] = df_results["qwen2.5:7b"].apply(
    lambda x: json.dumps(postprocess_json_output(x), ensure_ascii=False)
)

In [49]:
list(df_results["qwen2.5:7b"])[54+61]

'```json {   "tokens": ["รัก", "ตัวเอง", "ที่", "1", "ค่ะ", "🌸🌸"],   "rationales": [[4]],   "triggers": [     { "span": [4, 4], "rid": 0 }   ],   "sent_bias": 0 } ```'

In [45]:
import json
import re
from IPython.display import HTML, display

# --------- Clean JSON ---------
def extract_json(text):
    text = re.sub(r"```json|```", "", text).strip()
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        return m.group(0)
    return text

# --------- Highlight tokens with rationale + trigger ---------
def highlight_spans(tokens, rationales, triggers):
    colored = []
    
    # convert trigger spans into simple list like [ (start,end), ... ]
    trigger_spans = []
    for t in triggers:
        span = t.get("span")
        if span:
            trigger_spans.append((span[0], span[1]))

    for idx, tok in enumerate(tokens):
        color = None

        # Check trigger first (higher priority)
        for start, end in trigger_spans:
            if start <= idx <= end:
                color = "#ff4d4d"   # RED
                break
        
        # If not trigger, check rationale
        if color is None:
            for start, end in rationales:
                if start <= idx <= end:
                    color = "#ffcccc"  # LIGHT PINK
                    break

        if color:
            colored.append(f"<span style='background-color:{color}'>{tok}</span>")
        else:
            colored.append(tok)

    return " ".join(colored)

# --------- Visualize one row ---------
def visualize_row(raw_json_string):
    json_text = extract_json(raw_json_string)
    data = json.loads(json_text)

    tokens = data["tokens"]
    rationales = data.get("rationales", [])
    triggers = data.get("triggers", [])

    original = " ".join(tokens)
    highlighted = highlight_spans(tokens, rationales=rationales, triggers=triggers)

    html = f"""
    <div style="font-family:Trebuchet MS; font-size:16px; margin-bottom:25px;">
        <b>Original Sentence:</b><br>
        <p style="background:#f6f6f6; padding:8px;">{original}</p>

        <b>Highlighted (Pink=rationale, Red=trigger):</b><br>
        <p style="background:#f6f6f6; padding:8px;">{highlighted}</p>

        <b>Rationales:</b> {rationales}<br>
        <b>Triggers:</b> {triggers}
    </div>
    """
    display(HTML(html))

# --------- Visualize all rows in DF ---------
def visualize_all(df_results, col="qwen2.5:7b"):
    for i, item in enumerate(df_results[col].tolist()):
        display(HTML(f"<h3>Sentence {i+1}</h3>"))
        visualize_row(item)

In [48]:
visualize_all(df_results[54+61:1150], "qwen2.5:7b")

ValueError: not enough values to unpack (expected 2, got 1)